# Olist: orders by state

## Goal
Number of valid orders per state. The state is on the customers table, joined to orders on `customer_id`. The SQL version is in `sql/03_orders_by_state.sql`.

## Setup
Load orders and customers and rebuild `valid_orders` as in [02-monthly-orders](02-monthly-orders.ipynb).

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Project folder = the nearest parent that contains data/raw
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir())
DATA_DIR = ROOT / "data" / "raw"

orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")

# Valid orders (same definition as 02-monthly-orders)
orders["purchase_ts"] = pd.to_datetime(orders["order_purchase_timestamp"], errors="coerce")
valid_orders = orders[~orders["order_status"].isin(["canceled", "unavailable"])].copy()
valid_orders["purchase_month"] = valid_orders["purchase_ts"].dt.to_period("M")

print("Valid orders:", len(valid_orders), "| customer records:", len(customers))

Valid orders: 98207 | customer records: 99441


## Definitions
- **State**: `customers.customer_state`. Orders join to customers on `customer_id`, which is one per order, so the join cannot add rows. `customer_unique_id` is not the join key.
- **Orders**: the same valid orders as the monthly order count and monthly GMV.
- **Unit**: distinct `order_id`.
- **Checks**: the row count is unchanged by the join, no state is missing, and the states add up to 98,207.

In [2]:
# 1. Join orders to customers to get each order's state
orders_state = valid_orders.merge(
    customers[["customer_id", "customer_state"]], on="customer_id", how="left"
)

# 2. Join checks: same row count, unique orders, no missing state
print("Rows before join =", len(valid_orders), "| rows after join =", len(orders_state))
print("Distinct orders after join =", orders_state["order_id"].nunique())
print("Missing state =", orders_state["customer_state"].isna().sum())

Rows before join = 98207 | rows after join = 98207
Distinct orders after join = 98207
Missing state = 0


In [3]:
# 3. Distinct orders per state, descending
state_orders = (
    orders_state.groupby("customer_state")["order_id"].nunique()
    .sort_values(ascending=False)
    .rename("n_orders")
)
display(state_orders.head(10).to_frame())

# 4. Check: the states add up to the number of valid orders
print("States:", len(state_orders))
print("Sum of states:", state_orders.sum(), "| valid orders:", len(valid_orders), "| equal:", state_orders.sum() == len(valid_orders))
print("Share of the largest state:", round(state_orders.iloc[0] / state_orders.sum() * 100, 1), "%")

,n_orders
customer_state,
SP,41127
RJ,12698
MG,11496
RS,5417
PR,4983
SC,3600
BA,3344
DF,2121
ES,2018


States: 27
Sum of states: 98207 | valid orders: 98207 | equal: True
Share of the largest state: 41.9 %


## Findings
- 27 states; together they account for all 98,207 valid orders.
- São Paulo (SP) has 41,127 orders, 41.9% of the total. Rio de Janeiro (RJ) has 12,698 and Minas Gerais (MG) 11,496; the top three hold about two thirds of all orders.
- These are order counts by customer address, not by seller location.